# 11 · Django requests and Kafka events

**Level:** intermediate · **Time:** 45–60 minutes · **Needs:** running Kafka, notebooks 01–04.

Build an HTTP request → database → Kafka flow. Observe exactly when a database transaction
commits, then reproduce the gap between committing an order and publishing its event.
You will use the included Django application's real URL routing, views, and ORM through
Django's HTTP test client; starting a web server is optional.

**Run from top to bottom in a fresh kernel.** Each run creates its own database and topic.
Do not run the Django notebooks in one shared kernel: Django settings are process-wide.


In [ ]:
from pathlib import Path
import sys
from uuid import uuid4

ROOT = next((p for p in [Path.cwd(), *Path.cwd().parents]
             if (p / "pyproject.toml").exists() and (p / "labtools").is_dir()), None)
if ROOT is None:
    raise RuntimeError("Open Jupyter from the kafka-django workshop folder.")
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from labtools.kafka import (
    bootstrap_servers, client_config, require_broker, new_topic,
    producer, consumer, publish, poll_records, collect, decode,
)
RUN = uuid4().hex[:8]
print("Workshop:", ROOT.name, "| run:", RUN)


### Jupyter and synchronous Django code

Jupyter kernels may keep an event loop running even with `%autoawait off`. The next cell
installs `%%django_sync`: it executes the visible Python body on one dedicated synchronous
thread and waits for completion. All Django code cells below use that thread, so ORM
connection ownership and transaction boundaries remain consistent. Variables stay in the
notebook namespace, exceptions remain visible, and Django's async safety stays enabled.
The magic adds no business logic. In a terminal, run the same Python bodies normally.


In [ ]:
%autoawait off
sys.path.insert(0, str(ROOT / "django_demo"))
from notebook_support import install_sync_magic
DJANGO_EXECUTOR = install_sync_magic(get_ipython())
print("Synchronous Django cell executor installed")


## Step 1 — Start Django inside Jupyter

`%%django_sync` runs ORM operations without an active event loop on the database thread.
We do not disable Django's async safety protection. Migrations create four tables:
the original order, a durable outgoing event, an incoming event receipt, and a read model.
This notebook initially uses only `Order`.


In [ ]:
%%django_sync
LESSON = "11"

import os
import django
from django.conf import settings

if settings.configured:
    raise RuntimeError("Restart the kernel before running this notebook again; Django settings are process-wide.")
DATA_DIR = ROOT / ".lab-data"
DATA_DIR.mkdir(exist_ok=True)
DB_PATH = DATA_DIR / f"django-{LESSON}-{RUN}.sqlite3"
os.environ["DJANGO_DB_PATH"] = str(DB_PATH)
os.environ["DJANGO_SETTINGS_MODULE"] = "config.settings"
sys.path.insert(0, str(ROOT / "django_demo"))
django.setup()
from django.core.management import call_command
from django.db import transaction
from django.test import Client, override_settings
from django.utils import timezone
from decimal import Decimal
from unittest.mock import patch
from orders.models import Order, OutboxEvent, InboxEvent, OrderProjection
from orders.services import (
    order_event, validate_event, publish_event, apply_event,
    create_order_after_commit, create_order_with_outbox, publish_pending,
)
call_command("migrate", verbosity=0)
http = Client()
print("Isolated database:", DB_PATH.name)


In [ ]:
%%django_sync
from django.apps import apps
print([model.__name__ for model in apps.get_models()])
assert http.get("/health/").json()["status"] == "ok"


## Step 2 — Inspect the write boundary

The service creates the order inside `transaction.atomic()`, then registers a function
with `transaction.on_commit()`. The function runs after the outermost database transaction
commits. It cannot roll that committed transaction back.


In [ ]:
%%django_sync
import inspect
print(inspect.getsource(create_order_after_commit))


## Step 3 — Prepare the destination explicitly

Topic auto-creation is disabled in this workshop. One partition makes this first
demonstration easy to inspect; later services partition by order ID.


In [ ]:
%%django_sync
require_broker()
topic = new_topic("django-on-commit", partitions=1)
print(topic)


## Step 4 — Make a real Django HTTP request

A money amount is a string to avoid binary floating-point surprises. A UUID identifies
the order; another UUID identifies the event. `201` means the view returned successfully.
This teaching endpoint waits for its Kafka delivery callback, so broker latency affects
request latency. The outbox endpoint in notebook 13 will remove that dependency.


In [ ]:
%%django_sync
with override_settings(KAFKA_ORDERS_TOPIC=topic):
    response = http.post("/orders/on-commit/", {"customer": "Ada", "total": "49.90"}, content_type="application/json")
assert response.status_code == 201, response.content
result = response.json()
print(result)
assert Order.objects.filter(id=result["order_id"]).exists()


## Step 5 — Inspect what Kafka received

The message key is the order ID. The value is a versioned event envelope. The event
timestamp is UTC, and the money value remains a decimal string after serialization.


In [ ]:
%%django_sync
message = collect(topic, count=1)[0]
event = decode(message)
assert event["event_id"] == result["event_id"]
assert message.key().decode() == result["order_id"]
assert event["data"]["total"] == "49.90"
print(event)


## Step 6 — Reject invalid HTTP input before writing

A JSON float is rejected by the API contract. Validate before opening the transaction;
a rejected request should neither create an order nor publish a message.


In [ ]:
%%django_sync
before = Order.objects.count()
with override_settings(KAFKA_ORDERS_TOPIC=topic):
    bad = http.post("/orders/on-commit/", {"customer": "Ada", "total": 49.9}, content_type="application/json")
assert bad.status_code == 400
assert Order.objects.count() == before
print(bad.json())


## Step 7 — Observe deferred callbacks

This cell deliberately replaces only the network send with a spy. The Django database
transaction is real. The callback stays pending until the surrounding transaction exits.


In [ ]:
%%django_sync
with patch("orders.services.publish_event") as send:
    with transaction.atomic():
        deferred_order, _ = create_order_after_commit("Deferred", "5.00", topic=topic)
        assert send.call_count == 0
        print("Inside the transaction: callback has not run")
    assert send.call_count == 1
    print("After commit: callback ran once")


## Step 8 — Rollback removes the callback too

The intentional exception below is caught so Run All continues. Neither the order nor
its callback survives the rollback. This prevents an event describing an order that never existed.


In [ ]:
%%django_sync
before = Order.objects.count()
with patch("orders.services.publish_event") as send:
    try:
        with transaction.atomic():
            create_order_after_commit("Rolled back", "6.00", topic=topic)
            raise RuntimeError("simulated request failure")
    except RuntimeError:
        pass
    send.assert_not_called()
assert Order.objects.count() == before
print("Rollback removed both the order and the callback")


## Step 9 — Reproduce the lost-event window

Inject a broker failure after the database commits. This is a deterministic failure
simulation: we do not stop your broker. The order survives, but no durable outgoing
record exists. A process crash before the callback runs creates the same gap.


In [ ]:
%%django_sync
before = Order.objects.count()
with patch("orders.services.publish_event", side_effect=RuntimeError("simulated broker outage")):
    try:
        create_order_after_commit("Committed but unpublished", "7.00", topic=topic)
    except RuntimeError as exc:
        print(exc)
assert Order.objects.count() == before + 1
assert OutboxEvent.objects.count() == 0
print("The database committed. There is no durable retry record.")


## Step 10 — Optional live HTTP demonstration

In a separate terminal at the project root, first migrate the default database, then
start the development server bound to `127.0.0.1`:

```text
python django_demo/manage.py migrate
python django_demo/manage.py runserver 127.0.0.1:8000 --noreload
```

The terminal server uses its own database unless you set `DJANGO_DB_PATH` to this
notebook's printed path. Set `KAFKA_ORDERS_TOPIC` to the printed topic before starting
the server, then POST the same JSON with your preferred HTTP client. Stop with Ctrl+C.
The demo endpoints are unauthenticated and CSRF-exempt for local teaching; they are not
a deployment template. HTTP request retries also need a separate request-idempotency design.


## Practice and explain-back

1. Send `"1.001"`, `"NaN"`, and an empty customer. Predict each response before running it.
2. Move the intentional exception after the transaction block. Does the order remain?
3. Why does `on_commit()` prevent premature publication but not guarantee publication?
4. Draw the crash points between INSERT, COMMIT, PRODUCE, and acknowledgement.

**Solution hints:** invalid money/customer returns 400; exceptions after COMMIT cannot undo
it; a Python callback lives in process memory. Notebook 13 stores retryable intent in the
same database transaction. Notebook 12 first makes receiving duplicates safe.

Reference: [Django transaction callbacks](https://docs.djangoproject.com/en/5.2/topics/db/transactions/#performing-actions-after-commit)
and [IPython async context](https://docs.djangoproject.com/en/5.2/topics/async/#async-safety).
